# 🧪 LAB: Machine Learning to Predict Concrete Compressive Strength

In this lab, you will apply `PyTorch` to a real-world data problem. You will implement a Neural Network in `PyTorch` using the new functionalities introduced this week and compare its performance with a Support Vector Machine and Linear Regression.

Your goal is to predict the **compressive strength of concrete** (the outcome variable, *Y*) based on a set of input features (*X*), including:
**Cement, Blast Furnace Slag, Fly Ash, Water, Superplasticizer, Coarse Aggregate, Fine Aggregate, and Age.**

This dataset and problem are based on the following paper:

- I-Cheng Yeh, "Modeling of strength of high performance concrete using artificial neural networks," Cement and Concrete Research, Vol. 28, No. 12, pp. 1797-1808 (1998). [Link to the paper](yeh1998.pdf).


---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.

NAME(s) = "Laine Goldmacher, Max Magrill, Ashley Qin"

## 0. Overall Instructions 

In this lab, you will work with three models: a **Neural Network**, a **Support Vector Machine (SVM)**, and some form of **Linear Regression**. Each should meet the following requirements:

- **Do not tune the models**. For example, for the Neural Network, choose any number of layers and hidden units you prefer; for the SVM, select a kernel of your choice (e.g., linear, RBF, polynomial), etc.

- **Evaluate model performance using k-fold cross-validation**. That is, run each model using a chosen number of folds (*k*), and report the average performance across all folds. You may select the value of *k* that you want.

- **Be careful with data preprocessing**. Apply all preprocessing steps properly to avoid data leakage (e.g. standardizing features before data splitting)

## 1. Pre-implementation Group Discussion (15 points)

In your group, discuss, agree on, and elaborate the following points:

- **Descriptive analysis**. Identify what exploratory or descriptive analyses you can perform to better understand the relationship between each input variable and the outcome. Consider both visual (e.g., scatter plots, histograms) and statistical summaries.

- **Data preprocessing**. Decide what preprocessing steps are necessary given that you will be using neural networks. 

- **Model configuration**. Specify which cost function and output layer activation are most appropriate for your problem type (e.g., regression vs. classification). Explain your reasoning based on the nature of the outcome variable.

- **Performance evaluation**. Describe how you will perform k-fold cross-validation to assess your model’s performance. Include how you will divide the data, compute metrics across folds, and report the final averaged results.

Descriptive analysis: 
- Visually, we will create scatter plots and histograms to compare every input variable to y. This will allow us to visualize the spread of the data and how heavily the variables are correlated.
- Statistically, we will look at correlation matrices, as well as the mean, median, and standard deviation for each input variable. Since each input variable is numerical, these summaries will help us to analyze the distributions and relationships in the data.
Data Preprocessing:
- We will first apply a train/test split in order to separate our training and our holdout data. 
- We will next standardize the data before training each model because standardization ensures that features are on comparable scales, preventing features with larger numerical values from disproportionately influencing gradient updates. This improves the stability and convergence speed of stochastic gradient descent, making training more efficient. It should be fitted on the data before training each model to avoid data leakage; the scaler is fitted on the training data and then applied to both training and holdout sets.
Model configuration:
- 

In [6]:
# data loading
import pandas as pd
data = pd.read_excel('Concrete_Data.xls')
data.head()


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


## 2. Descriptive analysis (10 points)

Apply the descriptive analyses that your group agreed upon in the previous section.

**Load the dataset before beginning the descriptive analysis; you will also use it in the following sections.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# shorten the long column names so plots and tables are readable
data.columns = ['Cement', 'Slag', 'FlyAsh', 'Water', 'Superplasticizer',
                'CoarseAgg', 'FineAgg', 'Age', 'Strength']
features = data.columns[:-1]
target = 'Strength'

print(data.shape)
print("Missing values per column:\n", data.isna().sum())

### 2.1 Statistical summaries
Mean, median, and standard deviation of each variable (plus min/max to see the range).

In [ ]:
summary = pd.DataFrame({
    'mean': data.mean(),
    'median': data.median(),
    'std': data.std(),
    'min': data.min(),
    'max': data.max(),
    'pct_zero': (data == 0).mean() * 100,
}).round(2)
summary

### 2.2 Histograms
Distribution of each input variable and of the outcome.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.flat, data.columns):
    ax.hist(data[col], bins=30, edgecolor='black')
    ax.axvline(data[col].mean(), color='red', linestyle='--', label='mean')
    ax.axvline(data[col].median(), color='orange', linestyle='-', label='median')
    ax.set_title(col)
axes.flat[0].legend()
plt.suptitle('Distributions of inputs and outcome (Strength)')
plt.tight_layout()
plt.show()

### 2.3 Scatter plots: each input vs. Strength
Each panel shows one input against the outcome, with a fitted least-squares line and the Pearson correlation *r* in the title.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
for ax, col in zip(axes.flat, features):
    x, y = data[col], data[target]
    ax.scatter(x, y, alpha=0.4, s=12)
    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, slope * xs + intercept, color='red')
    ax.set_title(f'{col} (r = {x.corr(y):.2f})')
    ax.set_xlabel(col)
for ax in axes[:, 0]:
    ax.set_ylabel('Strength (MPa)')
plt.suptitle('Each input variable vs. compressive strength')
plt.tight_layout()
plt.show()

### 2.4 Correlation matrix
Pearson correlations between all variables, including each input's correlation with `Strength`.

In [ ]:
corr = data.corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()

# correlation of each input with the outcome, strongest first
corr[target].drop(target).sort_values(key=abs, ascending=False).round(3)

### 2.5 Key findings
- **Data quality:** 1030 observations, 8 numeric inputs, 1 numeric outcome, and no missing values. No imputation is needed.
- **Very different scales:** the inputs range from about 0–32 (Superplasticizer) up to 800–1145 (CoarseAgg), and their standard deviations differ by more than 10×. This supports our plan to standardize the features (fitted on training folds only) before training the neural network.
- **Skewed and zero-inflated inputs:** Slag (45% zeros), FlyAsh (55% zeros), and Superplasticizer (37% zeros) have large spikes at 0, because many mixtures leave those components out entirely. For these variables the mean is well above the median. Age is strongly right-skewed (median 28 days, max 365) and takes only a few discrete values (1, 3, 7, 14, 28, 56, 90, …).
- **Outcome:** Strength is roughly bell-shaped and slightly right-skewed (mean 35.8 vs. median 34.4 MPa, range 2.3–82.6). It is continuous, so this is a **regression** problem.
- **Relationships with Strength:** Cement has the strongest correlation (r = 0.50), followed by Superplasticizer (0.37) and Age (0.33). Water is negatively correlated (−0.29). Slag, FlyAsh, CoarseAgg, and FineAgg are only weakly correlated (|r| < 0.17).
- **Nonlinearity:** the scatter plots have a lot of spread around the linear fits, and no single input explains much of the variance. Age in particular looks nonlinear: strength rises quickly over the first ~28–56 days and then levels off. This suggests a nonlinear model such as a neural network or an RBF SVM should do better than plain linear regression.
- **Correlation among inputs:** Water and Superplasticizer are fairly strongly negatively correlated (r = −0.66), and Water and FineAgg are moderately negatively correlated (−0.45). This makes sense, since superplasticizer is used to reduce the water content. Some coefficients in a linear model may therefore be hard to interpret, but the correlation is not extreme.

## 3. Neural Network (40 points)

Implement a neural network as you did in the previous lab, but this time leverage the new functionalities introduced this week. Specifically:

- Your class implementing the neural network should inherit from nn.Module.

- Use `nn.Linear` for the linear transformations between layers, `nn.ReLU` for the hidden-layer activations, and the appropriate activation function for the output layer (e.g., `nn.Sigmoid` for binary classification).

- Use the appropriate cost function from `torch.nn` for your task (e.g., `nn.MSELoss`, `nn.BCELoss`).

- Use Adam from `torch.optim` as the optimizer.

- Use `DataLoader` to efficiently do batch processing. You may choose the batch size freely.

Once you have implemented this class, create an instance of it, then train and evaluate your model on the lab dataset.

**Remember to follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.**

In [4]:
# USE AS MANY CELLS AS NEEDED

## 4. Replication (10 points)

Replicate your neural network experiment using the corresponding model class from `scikit-learn`: https://scikit-learn.org/stable/api/sklearn.neural_network.html. Be sure to apply the same neural network architecture as well as the same procedures for model evaluation and preprocessing to ensure a fair comparison.

**Remember**: Follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.

In [5]:
# USE AS MANY CELLS AS NEEDED

## 5. Comparison (20 points)

Apply a Support Vector Machine (SVM) and Linear Regression model to the dataset, and compare their performance with that of your Neural Network.

Discuss any similarities or differences you observe in their results and provide possible explanations for these patterns.

Be sure to apply the k-fold cross-validation procedure to ensure a fair comparison across all models.

**Remember**: Follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.

In [6]:
# USE AS MANY CELLS AS NEEDED

## 6. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE